# Inspect a Qutes Bell circuit
Run from a source checkout. Install the backend as described in resources/README.md, then launch this notebook with that Python kernel.

In [ ]:
from pathlib import Path
import importlib.util
import tempfile
from qutes.compiler import compile_source
from qutes.codegen import write_python
from qutes.config import CompilerConfig

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'examples/language/41_bell_gates.qut').is_file())
tour = root / 'examples/language'
bell = (tour / '41_bell_gates.qut').read_text(encoding='utf-8')
source = 'input bool flip;\n' + bell.replace('return [', 'if (flip) { not(bob); }\nreturn [')
compiled = compile_source(source, inputs={'flip': False}, config=CompilerConfig.load(tour / 'qutes.toml'))
folder = tempfile.TemporaryDirectory()
path = Path(folder.name) / 'bell.py'
write_python(compiled, path)
spec = importlib.util.spec_from_file_location('bell', path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
circuit, layout = module.build()
print(circuit.draw(output='text'))

In [ ]:
for flip in (False, True):
    result = module.run(inputs={'flip': flip}, shots=64, seed=17)
    for row in result['outcomes']:
        alice, bob = row['value']
        assert (alice != bob) == flip
    print(flip, result['outcomes'])
folder.cleanup()

The input changes the compiled circuit. Equal bits and opposite bits are different correlation predictions. The seed makes this experiment repeatable within this backend version.